# 00 · Setup and tests

**Purpose:** check that everything is connected: Drive mounted, the latest code loaded (from GitHub plus the dev notebook's latest build in Drive), unit tests run, and the data files are where we expect. Run this first in any new session.

Code is written in **model_dev**, not here.

> **Data rules:** never print, display or save raw MIMIC-IV rows. Only aggregate counts and scores.
> Don't ask Colab's AI assistant about cells that touch the data. Never commit data or data outputs to GitHub.

## Setup (run every session)

Change `DRIVE_FOLDER` to where the team's shared folder appears in **your** Drive.
A folder someone shared with you shows under "Shared with me": right-click it and choose
**Organize → Add shortcut** to My Drive so Colab can see it.

In [ ]:
# --- Settings: edit these ---
DRIVE_FOLDER = "/content/drive/MyDrive/DL4H_TPC"  # shared team folder
BRANCH = "tpc-dev"
REPO_URL = "https://github.com/avnimohan9/PyHealth.git"

import os, sys
assert sys.version_info >= (3, 12), f"PyHealth needs Python 3.12+, this runtime has {sys.version}"

from google.colab import drive
drive.mount("/content/drive")

# Everything lives in the shared Drive folder
MIMIC4_ROOT = f"{DRIVE_FOLDER}/mimiciv/2.2"   # MIMIC-IV data (hosp/, icu/)
CACHE_DIR = f"{DRIVE_FOLDER}/cache"           # PyHealth's processed copy of the data
RESULTS_DIR = f"{DRIVE_FOLDER}/results"       # results spreadsheets
for d in (CACHE_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Get the latest code from GitHub (clone the first time, pull after that)
if os.path.exists("/content/PyHealth"):
    !git -C /content/PyHealth checkout -q -- .   # drop copies from last run; restored below
    !git -C /content/PyHealth pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} /content/PyHealth
%cd /content/PyHealth
!git log --oneline -1
!pip install -q -e .
# If "import pyhealth" fails right after the first install: Runtime -> Restart session, then rerun setup.

In [ ]:
# Use the latest code built in the model dev notebook (saved in Drive), if there is any
import shutil
for rel in ["pyhealth/models/tpc.py", "tests/core/test_tpc.py", "examples/mimic4_mortality_tpc.py"]:
    src = f"{DRIVE_FOLDER}/code/{rel}"
    if os.path.exists(src):
        shutil.copy(src, f"/content/PyHealth/{rel}")
        print("using the dev notebook's version of", rel)

## Unit tests
Should all pass (or skip while tests are still TODO).

In [ ]:
!python -m pytest tests/core/test_tpc.py -q

## Check the data files

In [ ]:
for f in ["hosp/patients", "hosp/admissions", "hosp/labevents", "hosp/d_labitems", "icu/icustays"]:
    found = [ext for ext in (".csv.gz", ".csv") if os.path.exists(f"{MIMIC4_ROOT}/{f}{ext}")]
    print(f"{f:20s}", "OK" if found else "MISSING")